<a href="https://colab.research.google.com/github/DURGAPRASAD-67/-ai-mentor-portfolio/blob/main/Day6B_PlacementProcessor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pydantic import BaseModel
from typing import List, Optional

In [3]:
class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [4]:
import requests
from bs4 import BeautifulSoup
import pathlib, json

def fetch_jd(url, max_chars=6000):
    """Fetch JD URL and return clean text. Returns None on block / failure."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        # Remove script and style tags
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:max_chars]
    except Exception as e:
        print(f'  Scrape failed for {url}: {e}')
        return None

# Test on one URL
test_url = 'TODO_REPLACE_WITH_ASSIGNED_URL'
text = fetch_jd(test_url)
if text:
    print(f'Got {len(text)} chars')
    print(text[:300])
else:
    print('Scrape blocked. Will use cached set.')

  Scrape failed for TODO_REPLACE_WITH_ASSIGNED_URL: Invalid URL 'TODO_REPLACE_WITH_ASSIGNED_URL': No scheme supplied. Perhaps you meant https://TODO_REPLACE_WITH_ASSIGNED_URL?
Scrape blocked. Will use cached set.


In [5]:
def normalise_jd(text: str) -> JD:
    """Send JD text to Gemini, get structured JD JSON back."""
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Extract a JD JSON from this text:\n\n{text}',
        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema(),
        },
    )
    return JD.model_validate_json(resp.text)

# Test on one JD text
if text:
    jd = normalise_jd(text)
    print(jd.model_dump_json(indent=2))

In [6]:
import json, pathlib

URLS = [
    # Paste your 5 assigned URLs here
    'https://www.tcsion.com/jobs/CLOUDNOW-TECHNOLOGIES-PRIVATE-LIMITED/Cloud-Engineer-6340/?utm_source=chatgpt.com',
    'https://in.indeed.com/viewjob?jk=215129ab637ada71&from=shareddesktop_copy',
    'https://in.indeed.com/viewjob?jk=f0e4fd8e63390917&from=shareddesktop_copy',
    'https://amazon.jobs/en/jobs/10386843/software-dev-engineer-ii',
    'https://in.indeed.com/viewjob?jk=daf426855f1c9175&from=shareddesktop_copy',
]

CACHE = pathlib.Path('/content/jds_cached.jsonl')
USE_CACHE = False   # set True if scraping is blocked

jds = []

if USE_CACHE and CACHE.exists():
    print(f'Using cached JDs from {CACHE}')
    for line in CACHE.read_text().splitlines():
        jds.append(JD.model_validate_json(line))
else:
    for url in URLS:
        text = fetch_jd(url)
        if text is None:
            continue
        try:
            jd = normalise_jd(text)
            jds.append(jd)
            print(f'  ✓ {jd.company} — {jd.role}')
        except Exception as e:
            print(f'  ✗ {url}: {e}')

print(f'\nProcessed {len(jds)} JDs')

# Inspect first 3
for jd in jds[:3]:
    print(f'\n{jd.company} - {jd.role}')
    print(f'  Must: {jd.must_have_skills}')
    print(f'  Nice: {jd.nice_to_have_skills}')
    print(f'  CGPA: {jd.min_cgpa}, LPA: {jd.package_lpa}')

  ✗ https://www.tcsion.com/jobs/CLOUDNOW-TECHNOLOGIES-PRIVATE-LIMITED/Cloud-Engineer-6340/?utm_source=chatgpt.com: name 'client' is not defined
  Scrape failed for https://in.indeed.com/viewjob?jk=215129ab637ada71&from=shareddesktop_copy: 403 Client Error: Forbidden for url: https://in.indeed.com/viewjob?jk=215129ab637ada71&from=shareddesktop_copy
  Scrape failed for https://in.indeed.com/viewjob?jk=f0e4fd8e63390917&from=shareddesktop_copy: 403 Client Error: Forbidden for url: https://in.indeed.com/viewjob?jk=f0e4fd8e63390917&from=shareddesktop_copy
  ✗ https://amazon.jobs/en/jobs/10386843/software-dev-engineer-ii: name 'client' is not defined
  Scrape failed for https://in.indeed.com/viewjob?jk=daf426855f1c9175&from=shareddesktop_copy: 403 Client Error: Forbidden for url: https://in.indeed.com/viewjob?jk=daf426855f1c9175&from=shareddesktop_copy

Processed 0 JDs


In [7]:
from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get('GOOGLE_API_KEY')
)

In [8]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hello"
)

print(response.text)

Hello! How can I help you today?


text = """
Company: TechNova Solutions

Role: Software Engineer

Location: Chennai

Package: 8 LPA

Skills:
Python
Java
SQL
AWS
Docker

Minimum CGPA: 7.0
"""

In [9]:
jd = normalise_jd(text)

print(jd.model_dump_json(indent=2))

{
  "company": "",
  "role": "",
  "must_have_skills": [],
  "nice_to_have_skills": [],
  "min_cgpa": null,
  "locations": [],
  "package_lpa": null
}


In [10]:
import pathlib

CACHE = pathlib.Path("jds_cached.jsonl")

jds = []

for line in CACHE.read_text().splitlines():
    jds.append(JD.model_validate_json(line))

print(f"Processed {len(jds)} JDs")

Processed 4 JDs


In [11]:
import json, pathlib

URLS = [
    # Paste your 5 assigned URLs here
    'https://www.tcsion.com/jobs/CLOUDNOW-TECHNOLOGIES-PRIVATE-LIMITED/Cloud-Engineer-6340/?utm_source=chatgpt.com',
    'https://in.indeed.com/viewjob?jk=215129ab637ada71&from=shareddesktop_copy',
    'https://in.indeed.com/viewjob?jk=f0e4fd8e63390917&from=shareddesktop_copy',
    'https://amazon.jobs/en/jobs/10386843/software-dev-engineer-ii',
    'https://in.indeed.com/viewjob?jk=daf426855f1c9175&from=shareddesktop_copy',
]

CACHE = pathlib.Path('/content/jds_cached.jsonl')
USE_CACHE = False   # set True if scraping is blocked

jds = []

if USE_CACHE and CACHE.exists():
    print(f'Using cached JDs from {CACHE}')
    for line in CACHE.read_text().splitlines():
        jds.append(JD.model_validate_json(line))
else:
    for url in URLS:
        text = fetch_jd(url)
        if text is None:
            continue
        try:
            jd = normalise_jd(text)
            jds.append(jd)
            print(f'  ✓ {jd.company} — {jd.role}')
        except Exception as e:
            print(f'  ✗ {url}: {e}')

print(f'\nProcessed {len(jds)} JDs')

# Inspect first 3
for jd in jds[:3]:
    print(f'\n{jd.company} - {jd.role}')
    print(f'  Must: {jd.must_have_skills}')
    print(f'  Nice: {jd.nice_to_have_skills}')
    print(f'  CGPA: {jd.min_cgpa}, LPA: {jd.package_lpa}')

  ✓ CLOUDNOW TECHNOLOGIES PRIVATE LIMITED — Cloud Engineer
  Scrape failed for https://in.indeed.com/viewjob?jk=215129ab637ada71&from=shareddesktop_copy: 403 Client Error: Forbidden for url: https://in.indeed.com/viewjob?jk=215129ab637ada71&from=shareddesktop_copy
  Scrape failed for https://in.indeed.com/viewjob?jk=f0e4fd8e63390917&from=shareddesktop_copy: 403 Client Error: Forbidden for url: https://in.indeed.com/viewjob?jk=f0e4fd8e63390917&from=shareddesktop_copy
  ✓ Amazon — Software Dev Engineer II
  Scrape failed for https://in.indeed.com/viewjob?jk=daf426855f1c9175&from=shareddesktop_copy: 403 Client Error: Forbidden for url: https://in.indeed.com/viewjob?jk=daf426855f1c9175&from=shareddesktop_copy

Processed 2 JDs

CLOUDNOW TECHNOLOGIES PRIVATE LIMITED - Cloud Engineer
  Must: ['Cloud Computing Fundamentals', 'AWS', 'Azure', 'GCP', 'Networking', 'Operating Systems', 'System Administration', 'Infrastructure as Code (IaC)', 'Terraform', 'CloudFormation', 'Scripting', 'Programming

In [12]:
OUT = pathlib.Path('data/jds.jsonl')
OUT.parent.mkdir(exist_ok=True)
with open(OUT, 'w') as f:
    for jd in jds:
        f.write(jd.model_dump_json() + '\n')
print(f'Wrote {len(jds)} JDs to {OUT}')

# Verify the file
with open(OUT) as f:
    for line in f:
        d = json.loads(line)
        print(f'  {d["company"]:20} | {d["role"]:30} | {len(d["must_have_skills"])} must-haves')

Wrote 2 JDs to data/jds.jsonl
  CLOUDNOW TECHNOLOGIES PRIVATE LIMITED | Cloud Engineer                 | 20 must-haves
  Amazon               | Software Dev Engineer II       | 16 must-haves
